report summary

1. T5
zeroshot, fewshot, full fine-tune
on spider + nba oracle

2. lora T5-base rank sweep
r4/8/16/32
on spider + nba oracle
can also add comparison to results from 1.

3. model comparison
lora_t5-base_r16, qlora, codet5p-200m_r16
on spider + nba oracle

4. nba adaptation
lora_t5-base_r16m, codet5p-200m_r16
nba oracle on n=0/10/20/70/all(150)

5. next step rag
baseline, rag(BM25 vs dense vs hybrid (top k = 1/3/5)), oracle on nall nba

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="talk")

EVAL_DIR = Path("../eval")
results = pd.read_csv(EVAL_DIR / "results_summary.csv")
spider = pd.read_csv(EVAL_DIR / "spider_summary.csv")


def _fmt_pct(v: float) -> str:
    return f"{100 * v:.1f}%"


def _prefer_seed42(df: pd.DataFrame, key_cols: list[str], run_col: str = "run") -> pd.DataFrame:
    if df.empty:
        return df.copy()
    x = df.copy()
    x["_has_s42"] = x[run_col].astype(str).str.contains(r"_s42_", regex=True).astype(int)
    x = x.sort_values([*key_cols, "_has_s42", "exact_acc"], ascending=[*([True] * len(key_cols)), False, False])
    x = x.drop_duplicates(subset=key_cols, keep="first").drop(columns=["_has_s42"])
    return x


def _annotate_ci_bars(ax, df: pd.DataFrame, xcol: str, ycol: str, low: str, high: str, order: list[str]):
    x_pos = {name: i for i, name in enumerate(order)}
    yerr = np.vstack([
        (df[ycol] - df[low]).to_numpy(),
        (df[high] - df[ycol]).to_numpy(),
    ])
    xs = [x_pos[v] for v in df[xcol].astype(str)]
    ax.errorbar(xs, df[ycol].to_numpy(), yerr=yerr, fmt="none", c="black", capsize=4, lw=1)


def _safe_ylim(df: pd.DataFrame, high_col: str, floor: float = 0.18) -> tuple[float, float]:
    if df.empty:
        return (0.0, 1.0)
    top = min(1.0, max(floor, float(df[high_col].max()) * 1.2))
    return (0.0, top)


## 1) T5: zeroshot / fewshot / full fine-tune（Spider + NBA Oracle）


In [ ]:
sec1_nba = results[(results["model"] == "t5-base") & (results["mode"] == "other")][[
    "run", "exact_acc", "exact_ci_low", "exact_ci_high", "exec_acc", "exec_ci_low", "exec_ci_high"
]].copy()
sec1_nba["setting"] = sec1_nba["run"].map({
    "baseline_nba_zeroshot_t5-base_test": "zero-shot",
    "baseline_nba_fewshot_t5-base_test": "few-shot",
})
full_nba = results[results["run"] == "full_t5-base_nba_oracle_test"][[
    "run", "exact_acc", "exact_ci_low", "exact_ci_high", "exec_acc", "exec_ci_low", "exec_ci_high"
]].copy()
if not full_nba.empty:
    full_nba["setting"] = "full fine-tune"
sec1_nba = pd.concat([sec1_nba.dropna(subset=["setting"]), full_nba], ignore_index=True)
sec1_nba["setting"] = pd.Categorical(sec1_nba["setting"], ["zero-shot", "few-shot", "full fine-tune"], ordered=True)
sec1_nba = sec1_nba.sort_values("setting")

sec1_spider = spider[spider["run"] == "full_t5-base_spider"][[
    "run", "exact_acc", "exact_ci_low", "exact_ci_high", "exec_acc", "exec_ci_low", "exec_ci_high"
]].copy()
if not sec1_spider.empty:
    sec1_spider["setting"] = "full fine-tune"

metric_specs = [
    ("exact_acc", "exact_ci_low", "exact_ci_high", "Exact Match"),
    ("exec_acc", "exec_ci_low", "exec_ci_high", "Execution Accuracy"),
]
fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)

for row, (metric, low, high, ylabel) in enumerate(metric_specs):
    if sec1_nba.empty:
        axes[row, 0].set_title(f"NBA Oracle {ylabel} (no data)")
        axes[row, 0].axis("off")
    else:
        order = ["zero-shot", "few-shot", "full fine-tune"]
        sns.barplot(data=sec1_nba, x="setting", y=metric, order=order, color="#4c72b0", ax=axes[row, 0])
        _annotate_ci_bars(axes[row, 0], sec1_nba, "setting", metric, low, high, order)
        axes[row, 0].set_title(f"T5-base on NBA Oracle - {ylabel}")
        axes[row, 0].set_ylabel(ylabel)
        axes[row, 0].set_ylim(*_safe_ylim(sec1_nba, high))

    if sec1_spider.empty:
        axes[row, 1].set_title(f"Spider {ylabel} (no data)")
        axes[row, 1].axis("off")
    else:
        order = ["full fine-tune"]
        sns.barplot(data=sec1_spider, x="setting", y=metric, order=order, color="#dd8452", ax=axes[row, 1])
        _annotate_ci_bars(axes[row, 1], sec1_spider, "setting", metric, low, high, order)
        axes[row, 1].set_title(f"T5-base on Spider - {ylabel}")
        axes[row, 1].set_ylabel(ylabel)
        axes[row, 1].set_ylim(*_safe_ylim(sec1_spider, high))

plt.show()


In [ ]:
msg1 = "- NBA Oracle: "
if sec1_nba.empty:
    msg1 += "缺少可用结果。"
else:
    best = sec1_nba.sort_values("exact_acc", ascending=False).iloc[0]
    msg1 += f"最佳是 **{best['setting']}**，Exact={_fmt_pct(best['exact_acc'])}。"
msg1 += "\n- Spider: 当前只有 full fine-tune，缺 zero/few-shot 对照。"
display(Markdown(msg1))


## 2) LoRA T5-base rank sweep（r4/r8/r16/r32）


In [ ]:
rank_pat = re.compile(r"lora_t5-base_r(\d+)")

sec2_nba = results[
    (results["model"] == "t5-base")
    & (results["method"] == "lora")
    & (results["mode"] == "oracle")
    & (results["n_train"].astype(str) == "0")
].copy()
sec2_nba["rank"] = pd.to_numeric(sec2_nba["run"].str.extract(rank_pat, expand=False), errors="coerce")
sec2_nba = _prefer_seed42(sec2_nba.dropna(subset=["rank"]), ["rank"]).sort_values("rank")

sec2_spider = spider[spider["run"].str.contains(r"^lora_t5-base_r\d+_spider$", regex=True)].copy()
sec2_spider["rank"] = pd.to_numeric(sec2_spider["run"].str.extract(rank_pat, expand=False), errors="coerce")
sec2_spider = sec2_spider.dropna(subset=["rank"]).sort_values("rank")

metric_specs = [
    ("exact_acc", "exact_ci_low", "exact_ci_high", "Exact Match"),
    ("exec_acc", "exec_ci_low", "exec_ci_high", "Execution Accuracy"),
]
fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
for row, (metric, low, high, ylabel) in enumerate(metric_specs):
    for col, (df, title) in enumerate([
        (sec2_nba, "NBA Oracle (n=0)"),
        (sec2_spider, "Spider"),
    ]):
        ax = axes[row, col]
        if df.empty:
            ax.set_title(f"{title} - {ylabel} (no data)")
            ax.axis("off")
            continue
        ax.plot(df["rank"], df[metric], marker="o", lw=2)
        ax.fill_between(df["rank"], df[low], df[high], alpha=0.18)
        ax.set_xticks(df["rank"])
        ax.set_xlabel("LoRA rank")
        ax.set_ylabel(ylabel)
        ax.set_ylim(*_safe_ylim(df, high))
        ax.set_title(f"{title} - {ylabel}")
plt.show()


In [ ]:
line2 = "- NBA Oracle: "
if sec2_nba.empty:
    line2 += "无 rank sweep 数据。"
else:
    best2 = sec2_nba.sort_values("exact_acc", ascending=False).iloc[0]
    line2 += f"最优 rank={int(best2['rank'])}，Exact={_fmt_pct(best2['exact_acc'])}。"
if sec2_spider.empty or sec2_spider["rank"].max() < 32:
    line2 += " 当前未包含 r32 的 Spider 对照。"
display(Markdown(line2))


## 3) 模型对比（LoRA t5-base r16 / QLoRA / LoRA codet5p-220m r16）


In [ ]:
records = []

nba_lora = results[results["run"] == "lora_t5-base_r16_nba_oracle_test"]
if not nba_lora.empty:
    records.append(("NBA Oracle", "LoRA t5-base r16", nba_lora.iloc[0]))

nba_qlora = _prefer_seed42(
    results[results["run"].str.contains(r"^qlora_t5-base.*nba_oracle_test$", regex=True)].copy(),
    ["model", "method", "mode", "n_train"],
)
if not nba_qlora.empty:
    records.append(("NBA Oracle", "QLoRA t5-base", nba_qlora.iloc[0]))

nba_codet = _prefer_seed42(
    results[results["run"].str.contains(r"^lora_codet5p-220m_r16.*nba_oracle_test$", regex=True)].copy(),
    ["model", "method", "mode", "n_train"],
)
if not nba_codet.empty:
    records.append(("NBA Oracle", "LoRA codet5p-220m r16", nba_codet.iloc[0]))

spi_lora = spider[spider["run"] == "lora_t5-base_r16_spider"]
if not spi_lora.empty:
    records.append(("Spider", "LoRA t5-base r16", spi_lora.iloc[0]))

spi_qlora = _prefer_seed42(spider[spider["run"].str.contains(r"^qlora_t5-base.*_spider$", regex=True)].copy(), ["model"])
if not spi_qlora.empty:
    records.append(("Spider", "QLoRA t5-base", spi_qlora.iloc[0]))

spi_codet = _prefer_seed42(spider[spider["run"].str.contains(r"^lora_codet5p-220m_r16.*_spider$", regex=True)].copy(), ["model"])
if not spi_codet.empty:
    records.append(("Spider", "LoRA codet5p-220m r16", spi_codet.iloc[0]))

sec3 = pd.DataFrame([
    {
        "dataset": d,
        "variant": m,
        "exact_acc": float(r["exact_acc"]),
        "exact_ci_low": float(r["exact_ci_low"]),
        "exact_ci_high": float(r["exact_ci_high"]),
        "exec_acc": float(r["exec_acc"]),
        "exec_ci_low": float(r["exec_ci_low"]),
        "exec_ci_high": float(r["exec_ci_high"]),
    }
    for d, m, r in records
])

metric_specs = [
    ("exact_acc", "exact_ci_high", "Exact Match"),
    ("exec_acc", "exec_ci_high", "Execution Accuracy"),
]
fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
for ax, (metric, high, ylabel) in zip(axes, metric_specs):
    if sec3.empty:
        ax.set_title(f"Model comparison - {ylabel} (no data)")
        ax.axis("off")
    else:
        sns.barplot(data=sec3, x="dataset", y=metric, hue="variant", ax=ax)
        ax.set_ylabel(ylabel)
        ax.set_ylim(*_safe_ylim(sec3, high))
        ax.set_title(f"Model Comparison - {ylabel}")
plt.show()


In [ ]:
if sec3.empty:
    display(Markdown("- 无法形成完整模型对比（数据不足）。"))
else:
    tops = sec3.sort_values("exact_acc", ascending=False).groupby("dataset").head(1)
    lines = [f"- {r['dataset']}: **{r['variant']}** 最优，Exact={_fmt_pct(r['exact_acc'])}" for _, r in tops.iterrows()]
    display(Markdown("\n".join(lines)))


## 4) NBA 适配曲线（n=0/10/20/70/all）


In [ ]:
sec4 = results[
    (results["mode"] == "oracle")
    & (results["n_train"].astype(str).isin(["0", "10", "20", "70", "all"]))
    & (
        ((results["model"] == "t5-base") & (results["run"].str.contains(r"^lora_t5-base_r16", regex=True)))
        | ((results["model"] == "codet5p-220m") & (results["run"].str.contains(r"^lora_codet5p-220m_r16", regex=True)))
    )
].copy()

sec4["variant"] = np.where(sec4["model"] == "t5-base", "LoRA t5-base r16", "LoRA codet5p-220m r16")
sec4 = _prefer_seed42(sec4, ["variant", "n_train"])
order = ["0", "10", "20", "70", "all"]
sec4["n_train"] = pd.Categorical(sec4["n_train"].astype(str), categories=order, ordered=True)
sec4 = sec4.sort_values(["variant", "n_train"])

metric_specs = [
    ("exact_acc", "exact_ci_low", "exact_ci_high", "Exact Match"),
    ("exec_acc", "exec_ci_low", "exec_ci_high", "Execution Accuracy"),
]
fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
for ax, (metric, low, high, ylabel) in zip(axes, metric_specs):
    if sec4.empty:
        ax.set_title(f"NBA adaptation curve - {ylabel} (no data)")
        ax.axis("off")
    else:
        xpos = np.arange(len(order))
        for name, g in sec4.groupby("variant"):
            g = g.sort_values("n_train")
            idx = [order.index(v) for v in g["n_train"].astype(str)]
            ax.plot(idx, g[metric], marker="o", lw=2, label=name)
            ax.fill_between(idx, g[low], g[high], alpha=0.15)
        ax.set_xticks(xpos)
        ax.set_xticklabels(order)
        ax.set_xlabel("n_train")
        ax.set_ylabel(ylabel)
        ax.set_ylim(*_safe_ylim(sec4, high))
        ax.set_title(f"NBA Adaptation by Training Size - {ylabel}")
        ax.legend()
plt.show()


In [ ]:
if not sec4.empty:
    at_all = sec4[sec4["n_train"].astype(str) == "all"].sort_values("exact_acc", ascending=False)
    if not at_all.empty:
        b4 = at_all.iloc[0]
        display(Markdown(f"- n=all 最优：**{b4['variant']}**，Exact={_fmt_pct(b4['exact_acc'])}。"))


## 5) RAG 下一步（baseline / rag@k / oracle）


In [ ]:
sec5_oracle_rag = results[
    (results["n_train"].astype(str) == "all")
    & (results["run"].str.contains("nba_rag_k", regex=False) | results["run"].str.contains("nba_oracle_test", regex=False))
    & (results["model"].isin(["t5-base", "codet5p-220m"]))
].copy()
sec5_oracle_rag = _prefer_seed42(sec5_oracle_rag, ["model", "run"])

sec5_oracle_rag["setting"] = np.where(
    sec5_oracle_rag["run"].str.contains("rag_k1", regex=False), "rag@k=1",
    np.where(
        sec5_oracle_rag["run"].str.contains("rag_k3", regex=False), "rag@k=3",
        np.where(sec5_oracle_rag["run"].str.contains("rag_k5", regex=False), "rag@k=5", "oracle"),
    ),
)

baseline_rows = results[
    (results["run"].isin(["baseline_nba_zeroshot_t5-base_test", "baseline_nba_fewshot_t5-base_test"]))
][["model", "exact_acc", "exact_ci_low", "exact_ci_high", "exec_acc", "exec_ci_low", "exec_ci_high"]].copy()
if not baseline_rows.empty:
    baseline_rows["model"] = "t5-base"
    baseline_rows["setting"] = "baseline"

sec5 = pd.concat(
    [
        sec5_oracle_rag[["model", "setting", "exact_acc", "exact_ci_low", "exact_ci_high", "exec_acc", "exec_ci_low", "exec_ci_high"]],
        baseline_rows[["model", "setting", "exact_acc", "exact_ci_low", "exact_ci_high", "exec_acc", "exec_ci_low", "exec_ci_high"]] if not baseline_rows.empty else pd.DataFrame(),
    ],
    ignore_index=True,
)

plot_order = ["baseline", "rag@k=1", "rag@k=3", "rag@k=5", "oracle"]
if not sec5.empty:
    sec5["setting"] = pd.Categorical(sec5["setting"], categories=plot_order, ordered=True)
    sec5 = sec5.sort_values(["model", "setting"])

metric_specs = [
    ("exact_acc", "exact_ci_high", "Exact Match"),
    ("exec_acc", "exec_ci_high", "Execution Accuracy"),
]
fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
for ax, (metric, high, ylabel) in zip(axes, metric_specs):
    if sec5.empty:
        ax.set_title(f"RAG next step - {ylabel} (no data)")
        ax.axis("off")
    else:
        sns.barplot(data=sec5, x="setting", y=metric, hue="model", ax=ax)
        ax.set_ylabel(ylabel)
        ax.set_ylim(*_safe_ylim(sec5, high))
        ax.set_title(f"Baseline vs RAG@k vs Oracle (NBA) - {ylabel}")
plt.show()


In [ ]:
notes = []
if not sec5.empty:
    rag_only = sec5[sec5["setting"].astype(str).str.startswith("rag@")]
    if not rag_only.empty:
        best5 = rag_only.sort_values("exact_acc", ascending=False).iloc[0]
        notes.append(f"- 当前最佳 RAG：**{best5['model']} / {best5['setting']}**，Exact={_fmt_pct(best5['exact_acc'])}。")
    notes.append("- 当前结果文件里没有 BM25 / dense / hybrid 的显式方法标签，建议后续在 run 命名里补充检索器类型。")

display(Markdown("\n".join(notes) if notes else "- RAG 对比数据不足。"))


---
以上图表与结论均来自 `eval/results_summary.csv` 与 `eval/spider_summary.csv`（bootstrap CI）。
